In [95]:
import pathlib as pl

cfg_nb = pl.Path("../../load-config.ipynb").resolve(strict=True)
%run $cfg_nb

_NB_SESSION = __session__
NB_NAME = pl.Path(_NB_SESSION).name
NB_PATH = pl.Path(_NB_SESSION).parent
NB_REL_PATH = NB_PATH.relative_to(CONFIG["project_repo"]).joinpath(NB_NAME)

PLOT_ROOT = CONFIG["plot_root"].joinpath(NB_PATH.stem)
NB_CACHE_FOLDER = CONFIG["nb_cache_folder"]

import collections as col
import dnaio
import io
import pandas as pd
import numpy as np
import numpy.ma as msk
import intervaltree as ivt

hg002y_file = NB_CACHE_FOLDER.joinpath("HG002-Y_T2Tv2_CP086569.2.fa.gz")

region_annotation_files = [
    CONFIG["project_repo"].joinpath(
        "annotation", "raw", "20250212_T2T-Y_annotations_from_Rhie_et_al_wRepeats.bed"
    ).resolve(strict=True),
    CONFIG["project_repo"].joinpath(
        "annotation", "raw", "20250212_T2T-Y_annotations_from_Rhie_et_al.bed"
    ).resolve(strict=True),
]

norm_names = [
    "HG002-Y_T2Tv2_regions_repeat-details.bed",
    "HG002-Y_T2Tv2_regions.bed"
]


def data_merger(current_data, new_data):

    if isinstance(current_data, str):
        assert isinstance(new_data, str)
        return [current_data, new_data]
    elif isinstance(current_data, list):
        assert isinstance(new_data, str)
        current_data.append(new_data)
        return current_data
    else:
        raise ValueError(f"{current_data} / {new_data}")


def merge_enclosed_repeat_units(unmerged_file, refseq_length):

    df = pd.read_csv(unmerged_file, sep="\t", header=0)
    df.sort_values("start", inplace=True)
    df.reset_index(drop=True, inplace=True)
    df["enum"] = np.arange(1, df.shape[0]+1, dtype=int)
    df.set_index("enum", inplace=True)
    
    region_cover = np.zeros(refseq_length, dtype=int)

    for row in df.itertuples():
        region_cover[row.start:row.end] = row.Index
   
    seqname = df["#chrom"].iloc[0]
    regions = []
    for uniq_value in np.unique(region_cover):
        masked_cover = msk.masked_not_equal(region_cover, uniq_value)
        region_name = df.loc[uniq_value, "name"]
        for region_slice in msk.clump_unmasked(masked_cover):
            start = int(region_slice.start)
            end = int(region_slice.stop)
            regions.append(
                (seqname, start, end, region_name)
            )
    regions = sorted(regions)
    regions = [
        (t[0], t[1], t[2], f"{pos:02}_{t[3]}", f"chrY_T2Tv2_{pos:02}_{t[3]}")
        for pos, t in enumerate(regions, start=1)
    ]
    regions = pd.DataFrame.from_records(
        regions,
        columns=["chrom", "start", "end", "name", "fasta_header"]
    )
    # skip over small regions
    regions["size"] = regions["end"] - regions["start"]
    regions.loc[regions["size"] < 1000, "fasta_header"] = "skip-short-seq"
    return regions
    

refseq_length = 62460029

with dnaio.open(hg002y_file) as fasta:
    for record in fasta:
        print(record.name)
        sequence = record.sequence
        assert len(sequence) == refseq_length

        fasta_names = []
        seqdb = io.StringIO()
        for annotation_file, norm_name in zip(region_annotation_files, norm_names):
            table = pd.read_csv(
                annotation_file, sep="\t", header=None,
                names=["chrom", "start", "end", "name"],
                usecols=["chrom", "start", "end", "name"],
                comment="#"
            )
            table.sort_values("start", inplace=True)

            # check for dup in coord space
            dups = table.duplicated(["start", "end"], keep=False)
            if dups.any():
                print(table.loc[dups, :])
                raise

            # check non-unique names
            name_counts = table["name"].value_counts()
            if name_counts.max() > 1:
                suffix_map = {
                    1
                }
                new_names = []
                seen = col.Counter()
                suffix_map = {
                    0: ".1",
                    1: ".2",
                    2: ".3"
                }
                for row in table.itertuples():
                    abd = seen[row.name]
                    if abd == 0 and name_counts[row.name] < 2:
                        new_names.append(row.name)
                    else:
                        assert name_counts[row.name] > 1, name_counts[row.name]
                        suffix = suffix_map[abd]
                        seen[row.name] += 1
                        new_names.append(f"{row.name}{suffix}")
                table["name"] = new_names
            
            table["fasta_header"] = table["name"].apply(lambda name: f"chrY_T2Tv2_{name.replace('_', '-')}")
    
            for row in table.itertuples():
                region_seq = sequence[row.start:row.end].upper()
                seqdb.write(f">{row.fasta_header}\n{region_seq}\n")

            out_file_annotation = annotation_file.parent.parent.joinpath("norm", norm_name)
            with open(out_file_annotation, "w") as bedlike:
                _ = bedlike.write("#")
                table.to_csv(bedlike, sep="\t", header=True, index=False)

            out_file_seqdb = NB_CACHE_FOLDER.joinpath(norm_name.replace(".bed", ".fasta"))
            with open(out_file_seqdb, "w") as fasta:
                _ = fasta.write(seqdb.getvalue())

            if "repeat-details" in norm_name:
                table = merge_enclosed_repeat_units(out_file_annotation, refseq_length)
                seqdb = io.StringIO()
                for row in table.itertuples():
                    region_seq = sequence[row.start:row.end].upper()
                    if "skip" in row.fasta_header:
                        continue
                    seqdb.write(f">{row.fasta_header}\n{region_seq}\n")

                disjoin_file = norm_name.replace("repeat-details", "repeat-disjoin")
                out_file_annotation = annotation_file.parent.parent.joinpath("norm", disjoin_file)
                with open(out_file_annotation, "w") as bedlike:
                    _ = bedlike.write("#")
                    table.to_csv(bedlike, sep="\t", header=True, index=False)

                out_file_seqdb = NB_CACHE_FOLDER.joinpath(disjoin_file.replace(".bed", ".fasta"))
                with open(out_file_seqdb, "w") as fasta:
                    _ = fasta.write(seqdb.getvalue())

        



gb|CP086569.2|:1-62460029 Homo sapiens isolate NA24385 chromosome Y
